In [ ]:
!rm -rf sample_data

In [ ]:
!rm -rf logs important logs_cleaned aggregated_results

In [ ]:
!tar xzf relevant_data_3.tar.gz

In [ ]:
import os
import re

# Configuration
INPUT_DIR = 'relevant_data'
OUTPUT_DIR = 'logs_cleaned'

# Define patterns: (tag, output_name, num_fields, has_extra_brackets)
PATTERNS = [
    ("[MPIAlltoallv]",    "baseline_mpi_alltoallv",       3, False),
    ("[ExcOr]",           "baseline_xor",                 3, False),
    ("[OMPI_pairwise]",   "baseline_ompi_pairwise",       3, False),
    ("[OMPI_linear]",     "baseline_ompi_linear",         3, False),
    ("[MPICH_scattered]", "baseline_mpich_scattered",     5, False),
    ("[Rbruckv]",         "parlogna",                     5, False),
    ("[TTPL]",            "parlinna_coalesced",           5, True),
    ("[TTPL_S1]",         "parlinna_staggered",           5, True),
    ("[ServletV1Slot]",   "parlinna_async",               5, False)
]

def clean_logs():
    if not os.path.exists(INPUT_DIR):
        print(f"Directory {INPUT_DIR} not found. Please check if extraction worked.")
        return

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    log_files = [f for f in os.listdir(INPUT_DIR) if f.endswith('.log')]

    for filename in log_files:
        input_path = os.path.join(INPUT_DIR, filename)
        output_path = os.path.join(OUTPUT_DIR, filename.replace('.log', '_cleaned.csv'))

        cleaned_lines = []

        with open(input_path, 'r') as f:
            for line in f:
                line = line.strip()
                if not line: continue

                for tag, algo_name, count, ignore_extra in PATTERNS:
                    if line.startswith(tag):
                        # Remove tag and strip whitespace
                        content = line[len(tag):].strip()

                        # Handle extra bracketed info for TTPL/TTPL_S1
                        if ignore_extra and '[' in content:
                            content = content.split('[')[0].strip()

                        # Split fields by comma
                        fields = [field.strip() for field in content.split(',') if field.strip()]

                        # Check if we have the correct amount of fields
                        if len(fields) == count:
                            csv_row = f"{algo_name}," + ",".join(fields)
                            cleaned_lines.append(csv_row)
                        break

        if cleaned_lines:
            with open(output_path, 'w') as f_out:
                f_out.write('\n'.join(cleaned_lines) + '\n')
            print(f"Processed {filename} -> {len(cleaned_lines)} rows extracted.")

clean_logs()

In [ ]:
import os
import pandas as pd

input_files = {f.replace('.log', '') for f in os.listdir(INPUT_DIR) if f.endswith('.log')}
output_files = {f.replace('_cleaned.csv', '') for f in os.listdir(OUTPUT_DIR) if f.endswith('_cleaned.csv')}

missing = input_files - output_files
if not missing:
    print(f"Verification Success: All {len(input_files)} log files have been cleaned.")
else:
    print(f"Warning: {len(missing)} files were not processed: {missing}")

# Inspect a sample file to verify content structure
sample_file = os.path.join(OUTPUT_DIR, os.listdir(OUTPUT_DIR)[0])
print(f"\nInspecting sample file: {sample_file}")
sample_df = pd.read_csv(sample_file, header=None, nrows=5)
display(sample_df)

In [ ]:
import os
from collections import defaultdict

AGG_DIR = 'aggregated_results'
os.makedirs(AGG_DIR, exist_ok=True)

# Header definitions based on algorithm requirements
HEADERS = {
    'baseline_mpi_alltoallv': 'name,nprocs,msg_size,max_elapsed',
    'baseline_ompi_linear': 'name,nprocs,msg_size,max_elapsed',
    'baseline_ompi_pairwise': 'name,nprocs,msg_size,max_elapsed',
    'baseline_xor': 'name,nprocs,msg_size,max_elapsed',
    'parlinna_async': 'name,nprocs,msg_size,block_size,radix,max_elapsed',
    'parlinna_coalesced': 'name,nprocs,msg_size,block_size,radix,max_elapsed',
    'parlinna_staggered': 'name,nprocs,msg_size,block_size,radix,max_elapsed',
    'parlogna': 'name,nprocs,msg_size,block_size,radix,max_elapsed'
}

algo_data = defaultdict(list)
cleaned_files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith('_cleaned.csv')]

for filename in cleaned_files:
    with open(os.path.join(OUTPUT_DIR, filename), 'r') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            parts = line.split(',', 1)
            if len(parts) > 1:
                algo_name = parts[0]
                content = parts[1]
                algo_data[algo_name].append(line) # Store the whole line including algo name

# Write out the aggregated files with headers
for algo_name, lines in algo_data.items():
    output_path = os.path.join(AGG_DIR, f"{algo_name}.csv")
    header = HEADERS.get(algo_name, "")
    with open(output_path, 'w') as f_out:
        if header:
            f_out.write(header + '\n')
        f_out.write('\n'.join(lines) + '\n')
    print(f"Aggregated {len(lines)} rows with headers for {algo_name} -> {output_path}")

In [ ]:
import pandas as pd

algorithm_dfs = {}

for algo_name in HEADERS.keys():
    file_path = os.path.join(AGG_DIR, f"{algo_name}.csv")
    if os.path.exists(file_path):
        # Read the header from the HEADERS dictionary
        column_names = HEADERS[algo_name].split(',')
        df = pd.read_csv(file_path, header=0, names=column_names)
        algorithm_dfs[algo_name] = df
        print(f"Loaded data for {algo_name}. Shape: {df.shape}")
    else:
        print(f"Warning: File not found for {algo_name} at {file_path}")

# Display the head of one of the DataFrames to verify
if algorithm_dfs:
    first_algo = next(iter(algorithm_dfs))
    print(f"\nDisplaying head of {first_algo} data:")
    display(algorithm_dfs[first_algo].head())

In [ ]:
import pandas as pd

parlogna_df = algorithm_dfs['parlogna']

# Filter out parlogna points after their highest peak for all nprocs
cleaned_parlogna_parts = []

for nproc in sorted(parlogna_df['nprocs'].unique()):
    subset = parlogna_df[parlogna_df['nprocs'] == nproc].copy()

    # Calculate mean per msg_size to find the robust peak
    means = subset.groupby('msg_size')['max_elapsed'].mean()
    if not means.empty:
        peak_msg_size = means.idxmax()
        print(f"For nprocs={nproc}, cutting off parlogna points after peak msg_size: {peak_msg_size}")
        # Keep only points up to (and including) the peak
        subset = subset[subset['msg_size'] <= peak_msg_size]

    cleaned_parlogna_parts.append(subset)

# Reconstruct the cleaned parlogna dataframe
algorithm_dfs['parlogna'] = pd.concat(cleaned_parlogna_parts, ignore_index=True)

# Re-calculate summary statistics for all algorithms based on the updated data
all_algo_summary_stats = pd.DataFrame()
for algo_name, df in algorithm_dfs.items():
    if 'nprocs' in df.columns and 'msg_size' in df.columns and 'max_elapsed' in df.columns:
        df['max_elapsed'] = pd.to_numeric(df['max_elapsed'], errors='coerce')
        df_cleaned = df.dropna(subset=['max_elapsed']).copy()
        summary = df_cleaned.groupby(['nprocs', 'msg_size'])['max_elapsed'].agg(['mean', 'median', 'std', 'min', 'max']).reset_index()
        summary['algorithm'] = algo_name
        all_algo_summary_stats = pd.concat([all_algo_summary_stats, summary], ignore_index=True)

# Re-apply buckets for consistency in downstream linear/log plots
def get_msg_size_bucket(msg_size):
    if msg_size < 1024: return 'Small (<1KB)'
    elif msg_size < 1024 * 1024: return 'Medium (1KB-1MB)'
    elif msg_size < 1024 * 1024 * 1024: return 'Large (1MB-1GB)'
    else: return 'Very Large (>=1GB)'

all_algo_summary_stats['msg_size_bucket'] = all_algo_summary_stats['msg_size'].apply(get_msg_size_bucket)
print("Summary statistics updated. All parlogna nprocs have been truncated after their highest peaks.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Identify all unique nprocs values across all dataframes
unique_nprocs = set()
for df in algorithm_dfs.values():
    if 'nprocs' in df.columns:
        unique_nprocs.update(df['nprocs'].unique())
unique_nprocs = sorted(list(unique_nprocs))

print(f"Unique 'nprocs' values found: {unique_nprocs}")

# List to store processed data for all algorithms and nprocs
all_combined_best_performance_data = []

for algo_name, df in algorithm_dfs.items():
    if 'nprocs' in df.columns and 'msg_size' in df.columns and 'max_elapsed' in df.columns:
        df_copy = df.copy() # Work on a copy to avoid SettingWithCopyWarning

        # Determine grouping columns for finding the best parameters
        group_cols_for_optimization = ['nprocs', 'msg_size']
        if 'block_size' in df_copy.columns:
            group_cols_for_optimization.append('block_size')
        if 'radix' in df_copy.columns:
            group_cols_for_optimization.append('radix')

        # Find the minimum max_elapsed for each unique combination of all relevant parameters
        # This effectively selects the 'best parameter' set (block_size, radix) for a given nprocs and msg_size
        min_elapsed_per_param_combo = df_copy.groupby(group_cols_for_optimization)['max_elapsed'].min().reset_index()

        # Now, from these optimized results, we need the minimum max_elapsed
        # for each (nprocs, msg_size) pair, effectively aggregating away block_size and radix
        final_best_for_algo = min_elapsed_per_param_combo.groupby(['nprocs', 'msg_size'])['max_elapsed'].min().reset_index()
        final_best_for_algo['name'] = algo_name # Add algorithm name for plotting
        all_combined_best_performance_data.append(final_best_for_algo)

if all_combined_best_performance_data:
    overall_best_performance_df = pd.concat(all_combined_best_performance_data, ignore_index=True)

    # Iterate through each unique nprocs value to create separate plots
    for nproc_val in unique_nprocs:
        data_for_current_nprocs = overall_best_performance_df[overall_best_performance_df['nprocs'] == nproc_val]

        if not data_for_current_nprocs.empty:
            plt.figure(figsize=(14, 8))
            sns.lineplot(data=data_for_current_nprocs, x='msg_size', y='max_elapsed', hue='name', marker='o')
            plt.title(f'Performance Comparison for nprocs = {nproc_val} (Best Parameters)')
            plt.xlabel('Message Size (msg_size)')
            plt.ylabel('Max Elapsed Time (seconds)')
            plt.xscale('log')
            plt.yscale('log')
            plt.xticks(rotation=45, ha='right')
            plt.grid(True, which="both", ls="--", c='0.7')
            plt.legend(title='Algorithm', bbox_to_anchor=(1.05, 1), loc='upper left')
            plt.tight_layout()
            plt.show()
        else:
            print(f"No data to plot for nprocs = {nproc_val}")
else:
    print("No best performance data generated for any algorithm to plot.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# We use the all_algo_summary_stats which contains mean, min, and max
unique_nprocs = sorted(all_algo_summary_stats['nprocs'].unique())
algorithms = all_algo_summary_stats['algorithm'].unique()

# Set a color palette for consistency
colors = sns.color_palette("husl", len(algorithms))
algo_color_map = dict(zip(algorithms, colors))

for nproc_val in unique_nprocs:
    plt.figure(figsize=(14, 8))
    data_nproc = all_algo_summary_stats[all_algo_summary_stats['nprocs'] == nproc_val]

    for algo in algorithms:
        algo_data = data_nproc[data_nproc['algorithm'] == algo].sort_values('msg_size')

        if not algo_data.empty:
            # Plot the mean line
            plt.plot(algo_data['msg_size'], algo_data['mean'],
                     label=algo, color=algo_color_map[algo], marker='.', linewidth=1.5)

            # Fill the range between min and max
            plt.fill_between(algo_data['msg_size'],
                             algo_data['min'],
                             algo_data['max'],
                             color=algo_color_map[algo],
                             alpha=0.2)

    plt.title(f'Performance Ranges (Min-Max) for nprocs = {nproc_val}')
    plt.xlabel('Message Size (bytes)')
    plt.ylabel('Max Elapsed Time (seconds)')
    plt.xscale('log')
    plt.yscale('log')
    plt.grid(True, which="both", ls="--", alpha=0.5)
    plt.legend(title='Algorithm', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Combine all dataframes into one long-form dataframe
combined_df_list = []
for algo_name, df in algorithm_dfs.items():
    temp_df = df.copy()
    temp_df['algorithm'] = algo_name
    combined_df_list.append(temp_df)

all_data_df = pd.concat(combined_df_list, ignore_index=True)
all_data_df['max_elapsed'] = pd.to_numeric(all_data_df['max_elapsed'], errors='coerce')

# Iterate through each unique process count
for nproc in sorted(all_data_df['nprocs'].unique()):
    plt.figure(figsize=(15, 7))
    subset = all_data_df[all_data_df['nprocs'] == nproc]

    sns.boxplot(data=subset, x='algorithm', y='max_elapsed', palette='Set3', hue='algorithm', legend=False)

    plt.title(f'Performance Distribution across Algorithms (nprocs={nproc})')
    plt.yscale('log')
    plt.ylabel('Max Elapsed Time (seconds) - Log Scale')
    plt.xticks(rotation=45)
    plt.grid(True, axis='y', linestyle='--', alpha=0.6)
    plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Grouping message sizes into buckets for clearer visualization
def get_msg_bucket(size):
    if size <= 1024: return 'Small (<=1KB)'
    if size <= 1024*1024: return 'Medium (1KB-1MB)'
    return 'Large (>1MB)'

all_algo_summary_stats['msg_bucket'] = all_algo_summary_stats['msg_size'].apply(get_msg_bucket)

# Create a FacetGrid to show scaling for each bucket
g = sns.FacetGrid(all_algo_summary_stats, col="msg_bucket", hue="algorithm",
                  col_order=['Small (<=1KB)', 'Medium (1KB-1MB)', 'Large (>1MB)'],
                  height=5, aspect=1.2, sharey=False)

g.map(sns.lineplot, "nprocs", "mean", marker="o")

g.set(xscale="log", yscale="log")
g.set_axis_labels("Number of Processes (nprocs)", "Mean Max Elapsed Time (s)")
g.add_legend(title="Algorithm")

plt.subplots_adjust(top=0.85)
g.fig.suptitle('Algorithm Scalability: Mean Performance vs. NProcs by Message Category')
plt.show()

In [ ]:
# Calculate the 'Scaling Slope' in log-log space
# slope = log(time2/time1) / log(nprocs2/nprocs1)
scalability_metrics = []

for (algo, bucket), group in all_algo_summary_stats.groupby(['algorithm', 'msg_bucket']):
    if len(group['nprocs'].unique()) > 1:
        # Fit a line to log(time) vs log(nprocs)
        log_n = np.log2(group['nprocs'])
        log_t = np.log2(group['mean'])
        slope, intercept = np.polyfit(log_n, log_t, 1)
        scalability_metrics.append({'algorithm': algo, 'msg_bucket': bucket, 'scaling_slope': slope})

scalability_df = pd.DataFrame(scalability_metrics).pivot(index='algorithm', columns='msg_bucket', values='scaling_slope')
print("Scaling Factor (Slope in log-log space): Closer to 0 is better (flat scaling)")
display(scalability_df)

In [ ]:
import pandas as pd

all_algo_summary_stats = pd.DataFrame()

for algo_name, df in algorithm_dfs.items():
    if 'nprocs' in df.columns and 'msg_size' in df.columns and 'max_elapsed' in df.columns:
        # Ensure 'max_elapsed' is numeric, coercing errors to NaN
        df['max_elapsed'] = pd.to_numeric(df['max_elapsed'], errors='coerce')

        # Drop rows where max_elapsed became NaN due to coercion
        df_cleaned = df.dropna(subset=['max_elapsed']).copy()

        # Group by relevant parameters and calculate summary statistics
        summary = df_cleaned.groupby(['nprocs', 'msg_size'])['max_elapsed'].agg(['mean', 'median', 'std', 'min', 'max']).reset_index()
        summary['algorithm'] = algo_name
        all_algo_summary_stats = pd.concat([all_algo_summary_stats, summary], ignore_index=True)

# Sort for better readability
all_algo_summary_stats = all_algo_summary_stats.sort_values(by=['nprocs', 'msg_size', 'algorithm'])

print("Summary Statistics for max_elapsed (grouped by nprocs, msg_size, and algorithm):")
display(all_algo_summary_stats.head(10))
display(all_algo_summary_stats.tail(10))


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# Define the bucketing function
def get_msg_size_bucket(msg_size):
    if msg_size < 1024:  # Up to 1KB
        return 'Small (<1KB)'
    elif msg_size < 1024 * 1024: # Up to 1MB
        return 'Medium (1KB-1MB)'
    elif msg_size < 1024 * 1024 * 1024: # Up to 1GB
        return 'Large (1MB-1GB)'
    else:
        return 'Very Large (>=1GB)'

# Re-apply bucketing to the summary stats
all_algo_summary_stats['msg_size_bucket'] = all_algo_summary_stats['msg_size'].apply(get_msg_size_bucket)

# Sort the buckets for logical ordering in the grid
custom_bucket_order = ['Small (<1KB)', 'Medium (1KB-1MB)', 'Large (1MB-1GB)', 'Very Large (>=1GB)']
bucket_order = [b for b in custom_bucket_order if b in all_algo_summary_stats['msg_size_bucket'].unique()]

# Set up the FacetGrid
g = sns.FacetGrid(
    all_algo_summary_stats,
    col="msg_size_bucket",
    col_order=bucket_order,
    hue="algorithm",
    height=5,
    aspect=1.2,
    sharey=False
)

# Map the lineplot to the grid
g.map(sns.lineplot, "nprocs", "mean", marker="o")

# Adjust formatting
g.set_titles("Message Size: {col_name}")
g.set_axis_labels("Number of Processes (nprocs)", "Mean Max Elapsed (s)")
g.set(xscale="log", yscale="log")
g.add_legend(title="Algorithm")

for ax in g.axes.flat:
    ax.grid(True, which="both", ls="--", alpha=0.5)
    nproc_ticks = sorted(all_algo_summary_stats['nprocs'].unique())
    ax.set_xticks(nproc_ticks)
    ax.set_xticklabels(nproc_ticks)

plt.subplots_adjust(top=0.85)
g.fig.suptitle('Scalability Analysis: Performance vs. NProcs across Message Size Buckets')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Get unique nprocs values from the summary statistics dataframe
unique_nprocs_summary = all_algo_summary_stats['nprocs'].unique()
unique_nprocs_summary.sort()

for nproc_val in unique_nprocs_summary:
    # Filter data for the current nprocs value
    data_for_nprocs = all_algo_summary_stats[all_algo_summary_stats['nprocs'] == nproc_val]

    # Calculate the overall mean max_elapsed for each algorithm for this nprocs value
    # by averaging the 'mean' column (which is already the mean over runs for each msg_size)
    overall_mean_performance = data_for_nprocs.groupby('algorithm')['mean'].mean().reset_index()
    overall_mean_performance = overall_mean_performance.sort_values(by='mean', ascending=True)

    if not overall_mean_performance.empty:
        plt.figure(figsize=(12, 7))
        # Fixed FutureWarning: Assign the `x` variable to `hue` and set `legend=False`
        sns.barplot(x='algorithm', y='mean', data=overall_mean_performance, hue='algorithm', palette='viridis', legend=False)
        plt.title(f'Mean Performance (max_elapsed) for nprocs = {nproc_val} (Averaged Across Message Sizes)')
        plt.xlabel('Algorithm')
        plt.ylabel('Average Mean Max Elapsed Time (seconds)')
        plt.yscale('log') # Use log scale for y-axis as max_elapsed can vary widely
        plt.xticks(rotation=45, ha='right')
        plt.grid(True, which="both", ls="--", c='0.7')
        plt.tight_layout()
        plt.show()
    else:
        print(f"No data to plot for nprocs = {nproc_val} in summary statistics.")


In [ ]:
import numpy as np

# Define logarithmic message size buckets
def get_msg_size_bucket(msg_size):
    if msg_size < 1024:  # Up to 1KB
        return 'Small (<1KB)'
    elif msg_size < 1024 * 1024: # Up to 1MB
        return 'Medium (1KB-1MB)'
    elif msg_size < 1024 * 1024 * 1024: # Up to 1GB
        return 'Large (1MB-1GB)'
    else:
        return 'Very Large (>=1GB)'

# Apply the bucketing function to the summary statistics DataFrame
all_algo_summary_stats['msg_size_bucket'] = all_algo_summary_stats['msg_size'].apply(get_msg_size_bucket)

print("DataFrame with message size buckets:")
display(all_algo_summary_stats.head())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Get unique message size buckets and sort them for consistent plotting order
unique_msg_size_buckets = all_algo_summary_stats['msg_size_bucket'].unique()
# Define a custom order for buckets if default sorting is not logical
custom_bucket_order = ['Small (<1KB)', 'Medium (1KB-1MB)', 'Large (1MB-1GB)', 'Very Large (>=1GB)']
unique_msg_size_buckets = sorted(unique_msg_size_buckets, key=lambda x: custom_bucket_order.index(x))

# Plot performance for each message size bucket
for bucket in unique_msg_size_buckets:
    plt.figure(figsize=(14, 8))
    data_for_bucket = all_algo_summary_stats[all_algo_summary_stats['msg_size_bucket'] == bucket]

    # Group by nprocs and algorithm to get the mean max_elapsed for plotting
    # This aggregates multiple msg_sizes within the same bucket for each nprocs/algo combo
    plot_data = data_for_bucket.groupby(['nprocs', 'algorithm'])['mean'].mean().reset_index()

    sns.lineplot(data=plot_data, x='nprocs', y='mean', hue='algorithm', marker='o')
    plt.title(f'Mean Performance vs. NProcs for Message Size: {bucket}')
    plt.xlabel('Number of Processes (nprocs)')
    plt.ylabel('Average Mean Max Elapsed Time (seconds)')
    plt.xscale('log')
    plt.yscale('log') # Log scale is often appropriate for performance data
    plt.xticks(plot_data['nprocs'].unique(), rotation=45, ha='right') # Show specific nprocs values
    plt.grid(True, which="both", ls="--", c='0.7')
    plt.legend(title='Algorithm', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

algorithms_to_compare = ['parlinna_coalesced', 'parlinna_async']

# Filter the overall best performance DataFrame for only the algorithms we want to compare
comparison_df = overall_best_performance_df[
    overall_best_performance_df['name'].isin(algorithms_to_compare)
].copy()

# Ensure 'nprocs' and 'msg_size' are numeric for proper plotting
comparison_df['nprocs'] = pd.to_numeric(comparison_df['nprocs'])
comparison_df['msg_size'] = pd.to_numeric(comparison_df['msg_size'])

# Identify all unique nprocs values from the comparison data
unique_nprocs_comparison = sorted(comparison_df['nprocs'].unique())

if not comparison_df.empty:
    for nproc_val in unique_nprocs_comparison:
        data_for_current_nprocs = comparison_df[comparison_df['nprocs'] == nproc_val]

        if not data_for_current_nprocs.empty:
            plt.figure(figsize=(14, 8))
            sns.lineplot(data=data_for_current_nprocs, x='msg_size', y='max_elapsed', hue='name', marker='o')
            plt.title(f'Performance Comparison: {', '.join(algorithms_to_compare)} for nprocs = {nproc_val}')
            plt.xlabel('Message Size (msg_size)')
            plt.ylabel('Max Elapsed Time (seconds)')
            plt.xscale('log')
            plt.yscale('log')
            plt.xticks(rotation=45, ha='right')
            plt.grid(True, which="both", ls="--", c='0.7')
            plt.legend(title='Algorithm', bbox_to_anchor=(1.05, 1), loc='upper left')
            plt.tight_layout()
            plt.show()
        else:
            print(f"No data to plot for nprocs = {nproc_val} for the selected algorithms.")
else:
    print(f"No performance data found for algorithms: {', '.join(algorithms_to_compare)}.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

algorithms_to_compare = ['parlinna_coalesced', 'parlinna_async']

# Filter the summary statistics DataFrame for the two algorithms we want to compare
scaling_comparison_df = all_algo_summary_stats[
    all_algo_summary_stats['algorithm'].isin(algorithms_to_compare)
].copy()

# Get unique message size buckets and sort them for consistent plotting order
# Re-using the custom order for clarity
custom_bucket_order = ['Small (<1KB)', 'Medium (1KB-1MB)', 'Large (1MB-1GB)', 'Very Large (>=1GB)']
unique_msg_size_buckets_filtered = sorted(
    scaling_comparison_df['msg_size_bucket'].unique(),
    key=lambda x: custom_bucket_order.index(x) if x in custom_bucket_order else len(custom_bucket_order)
)

if not scaling_comparison_df.empty:
    for bucket in unique_msg_size_buckets_filtered:
        plt.figure(figsize=(14, 8))
        data_for_bucket = scaling_comparison_df[scaling_comparison_df['msg_size_bucket'] == bucket]

        # Plot mean max_elapsed vs. nprocs for each algorithm within the bucket
        sns.lineplot(data=data_for_bucket, x='nprocs', y='mean', hue='algorithm', marker='o')
        plt.title(f'Scalability Comparison for Message Size: {bucket}')
        plt.xlabel('Number of Processes (nprocs)')
        plt.ylabel('Average Mean Max Elapsed Time (seconds)')
        plt.xscale('log')
        plt.yscale('log') # Log scale is often appropriate for performance data
        plt.xticks(data_for_bucket['nprocs'].unique(), rotation=45, ha='right') # Show specific nprocs values
        plt.grid(True, which="both", ls="--", c='0.7')
        plt.legend(title='Algorithm', bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout()
        plt.show()
else:
    print(f"No summary data found for algorithms: {', '.join(algorithms_to_compare)} to perform scalability comparison.")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Using the previously calculated best-performance data
# overall_best_performance_df should contain the min elapsed for each algo/nproc/msg combo

if 'overall_best_performance_df' in globals():
    unique_nprocs = sorted(overall_best_performance_df['nprocs'].unique())

    for nproc_val in unique_nprocs:
        data_for_current_nprocs = overall_best_performance_df[overall_best_performance_df['nprocs'] == nproc_val]

        if not data_for_current_nprocs.empty:
            plt.figure(figsize=(14, 8))
            # Using linear scale for Y-axis as requested
            sns.lineplot(data=data_for_current_nprocs, x='msg_size', y='max_elapsed', hue='name', marker='o')

            plt.title(f'Performance Comparison (Linear Scale) for nprocs = {nproc_val}')
            plt.xlabel('Message Size (bytes)')
            plt.ylabel('Max Elapsed Time (seconds)')

            # We keep X as log because the msg_sizes span several orders of magnitude
            plt.xscale('log')
            plt.grid(True, which="both", ls="--", c='0.7')
            plt.legend(title='Algorithm', bbox_to_anchor=(1.05, 1), loc='upper left')
            plt.tight_layout()
            plt.show()
        else:
            print(f'No data to plot for nprocs = {nproc_val}')
else:
    print('Variable overall_best_performance_df not found. Please ensure the optimization cells were run.')

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Filter for algorithms that have these parameters
param_algos = ['parlinna_async', 'parlinna_coalesced', 'parlinna_staggered', 'parlogna']

# Configuration to visualize
target_nprocs = 128

for algo in param_algos:
    if algo in algorithm_dfs:
        df = algorithm_dfs[algo]
        # We average across all message sizes for this process count to see general parameter behavior
        subset = df[df['nprocs'] == target_nprocs]

        if not subset.empty:
            # Pivot to create a matrix of block_size vs radix
            pivot = subset.pivot_table(index='block_size', columns='radix', values='max_elapsed', aggfunc='mean')

            plt.figure(figsize=(10, 8))
            sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlGnBu_r')
            plt.title(f'Heatmap: {algo} (nprocs={target_nprocs})\nMean Max Elapsed Time')
            plt.xlabel('Radix')
            plt.ylabel('Block Size')
            plt.show()
        else:
            print(f'No data available for {algo} at nprocs={target_nprocs}')

In [ ]:
import pandas as pd

# Define the bucketing function for message sizes
def get_msg_size_bucket(msg_size):
    if msg_size < 1024:  # Up to 1KB
        return 'Small (<1KB)'
    elif msg_size < 1024 * 1024: # Up to 1MB
        return 'Medium (1KB-1MB)'
    elif msg_size < 1024 * 1024 * 1024: # Up to 1GB
        return 'Large (1MB-1GB)'
    else:
        return 'Very Large (>=1GB)'

# Define the algorithms for which we want to find best parameters
param_algos = [
    'parlinna_async',
    'parlinna_coalesced',
    'parlinna_staggered',
    'parlogna'
]

best_parameters_list = []

for algo_name in param_algos:
    if algo_name in algorithm_dfs:
        df = algorithm_dfs[algo_name].copy()
        # Ensure max_elapsed is numeric
        df['max_elapsed'] = pd.to_numeric(df['max_elapsed'], errors='coerce')
        df = df.dropna(subset=['max_elapsed'])

        # Apply message size bucketing
        df['msg_size_bucket'] = df['msg_size'].apply(get_msg_size_bucket)

        # Group by nprocs and msg_size_bucket to find the best block_size and radix
        # for each combination. idxmin() will return the index of the first minimum, keeping
        # the specific msg_size that caused it within that bucket.
        grouped = df.loc[df.groupby(['nprocs', 'msg_size_bucket'])['max_elapsed'].idxmin()]

        # Select relevant columns, including msg_size_bucket
        best_params_subset = grouped[['nprocs', 'msg_size_bucket', 'msg_size', 'block_size', 'radix', 'max_elapsed']].copy()
        best_params_subset['algorithm'] = algo_name
        best_parameters_list.append(best_params_subset)

    else:
        print(f"Warning: DataFrame for {algo_name} not found in algorithm_dfs.")

if best_parameters_list:
    best_parameters_df = pd.concat(best_parameters_list, ignore_index=True)
    # Sort for better readability, including the new bucket column
    best_parameters_df = best_parameters_df.sort_values(by=['algorithm', 'nprocs', 'msg_size_bucket', 'msg_size'])
    print("Table showing the best parameters (block_size, radix) for each algorithm, nprocs, and message size bucket:")
    display(best_parameters_df)
else:
    print("No relevant data found to generate the best parameters table.")